# 04-llmlingua — 작은 모델이 토큰을 골라 버립니다

앞선 랩들과 판단 주체가 다릅니다.

| 랩 | 누가 판단하나 | 언어 영향 |
|---|---|---|
| `01` 무손실 | 규칙 | 없음 |
| `02` 참조핸들 | 겹침 점수 | 적음 |
| `03` 요약 | **큰 모델**이 다시 씁니다 | 적음 |
| **`04`** 프루닝 | **작은 모델**이 토큰마다 판정합니다 | **큽니다** |

중요도를 매기는 모델이 작아서, 그 모델이 약한 언어에서는 성능이 떨어집니다.
그래서 이 랩만 **한·영 이중언어 코퍼스**를 씁니다.

> **결론을 미리 말씀드리면** — 세 변형 중 `v2` 만 쓸 만했고,
> 같은 설정에서 **한국어가 영어보다 먼저 무너집니다.**

## ⚠️ 처음 실행은 오래 걸립니다

모델을 내려받습니다. `v2` 약 700MB, `v1`/`long` 약 1GB 입니다.
이 랩은 **전용 가상환경**을 쓰므로 커널을 `labs/04-llmlingua/.venv` 로
잡아주세요.

## 1. kit 과 어댑터 불러오기

In [ ]:
import sys
from pathlib import Path

LAB = Path.cwd().resolve()
LABS = LAB.parents[0]                  # labs/<이 랩> -> labs
sys.path.insert(0, str(LABS))
sys.path.insert(0, str(LAB))           # 이 랩의 모듈(transforms, blocks 등)

# 노트북을 켜 둔 채로 저장소를 갱신하면 커널이 **예전 코드를 물고 있습니다.**
# 그러면 새로 생긴 함수가 없다는 에러(AttributeError)가 나는데, 원인이 코드가
# 아니라 커널이라 찾기가 어렵습니다. 그래서 이 셀을 돌릴 때마다 새로 읽습니다.
_stale = [m for m in list(sys.modules)
          if m == "kit" or m.startswith("kit.")
          or m in ("transforms", "blocks", "summarize", "compress")]
for _m in _stale:
    del sys.modules[_m]

from kit import VERSION, config as C, dataset, env, metrics, tokens as T
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
env.load(verbose=True)

RUNS = LABS.parent / "runs"
print("kit", VERSION, "· 랩", LAB.name)
if _stale:
    print(f"모듈 {len(_stale)}개를 새로 읽었습니다 — 커널에 남아 있던 예전 코드를 지웠습니다")

import lingua as L
from compress import compress
from kit.metrics import survival

table(
    ["변형", "무엇이 다른가", "필요한 것"],
    [["v1", "토큰별 정보량으로 프루닝", "인과 LM"],
     ["long", "질문을 주고 문단별 중요도를 함께 봄", "인과 LM + 질문"],
     ["v2", "분류 모델이 토큰을 남길지 판정", "전용 인코더"]],
    align=["left", "left", "left"],
    title="LLMLingua 3형제 — 같은 클래스, 다른 파라미터",
    note="논문은 v1/long 에 7B 를 썼습니다. 여기서는 0.5B 를 쓰므로 "
         "그만큼 결과가 나쁩니다. 아래에서 그 영향을 직접 봅니다.",
)
for k, v in L.DEFAULT_MODEL.items():
    print(f"  {k:5s} {v}")

## 지표 두 가지 — 표를 읽기 전에

| 이름 | 무엇을 재나 |
|---|---|
| **절감** | 토큰이 얼마나 줄었나 |
| **보존율** | 답에 꼭 필요한 문자열(`must_include`)이 압축 후에도 남은 비율 |

보존율 예시입니다.

```
질문        3월 결제 총액과 환불액은?
필요한 것    ["32,450,000", "1,280,500"]   ← 2개

압축 후 2개 다 남음  → 100%
1개만 남음          →  50%
```

아래 표에서 **`전체` · `한국어` · `영어` 는 전부 같은 보존율**입니다.
전체는 12건 평균이고, 나머지는 그중 해당 언어만 골라 낸 평균입니다.

> 지금은 한·영이 6건씩 같아서 전체가 두 언어의 가운데값과 일치합니다.
> **건수가 달라지면 많은 쪽으로 기웁니다.** 한국어 서비스에 쓰실 거라면
> 전체가 아니라 **한국어 열**을 기준으로 보세요.

## 2. 코퍼스 — 같은 사실을 한국어와 영어로

번역이 아니라 **같은 사실을 담은 쌍**입니다. `pair_id` 로 묶여 있어
언어별 차이를 케이스 단위로 볼 수 있습니다.

In [ ]:
cases = dataset.load("../data/sample-bilingual")
counter = T.make_counter({"mode": "local"}, "gpt-5.4")

ko = [c for c in cases if c.meta["lang"] == "ko"]
en = [c for c in cases if c.meta["lang"] == "en"]

table(
    ["언어", "건수", "문자", "토큰", "문자당 토큰"],
    [[lg, len(xs), f"{sum(len(c.text) for c in xs):,}",
      f"{sum(counter(c.text) for c in xs):,}",
      f"{sum(counter(c.text) for c in xs) / sum(len(c.text) for c in xs):.2f}"]
     for lg, xs in [("한국어", ko), ("영어", en)]],
    align=["left", "right", "right", "right", "right"],
    title="이중언어 코퍼스",
    note="같은 내용인데 한국어가 문자당 토큰을 더 씁니다. 압축이 더 절실한 "
         "쪽인데, 아래에서 보시면 품질은 더 나쁩니다.",
)

c = ko[0]
print(f"[{c.id}] {c.question}")
print(f"  {c.text[:70]}…")
print(f"  정답 문자열 {c.must_include}")

## 3. 세 변형이 같은 문장을 어떻게 다루나

한 케이스에 셋을 다 걸어 봅니다. **처음 실행하면 모델 세 개를 받느라
몇 분 걸립니다.**

In [ ]:
probe = [c for c in cases if c.id == "en-01"][0]
print("원문:", probe.text[:100], "…")
print()

rows = []
for v in ["v1", "long", "v2"]:
    out, meta = compress(probe.text, question=probe.question, variant=v,
                         rate=0.5, force_reserve_digit=True)
    rows.append([v, f"{counter(probe.text)} → {counter(out)}",
                 pct(1 - counter(out) / counter(probe.text)),
                 pct(survival(out, probe.must_include)),
                 out[:52].replace(chr(10), " ")])
    print(f"  {v} 완료", flush=True)

table(
    ["변형", "토큰", "절감", "보존율", "결과 앞부분"],
    rows,
    align=["left", "right", "right", "right", "left"],
    title=f"{probe.id} · 질문: {probe.question}",
    note="v1/long 은 숫자 중간이 잘립니다. 토큰 단위로 자르기 때문입니다.",
)

`v1` 과 `long` 의 결과를 보시면 `32,450,000` 이 `32,450,00RW` 처럼
**숫자 중간에서 잘립니다.** 토큰 단위로 버리기 때문입니다.

`v2` 가 안전한 이유는 분류 모델이 단어에 가까운 단위로 판정해서 숫자를
통째로 남기기 때문입니다.

> 이건 **작은 모델을 써서** 생기는 문제입니다. "LongLLMLingua 가 나쁘다"
> 가 아니라 "작은 순위 모델로는 못 쓴다" 로 읽어주세요.

## 4. 조용히 무시되는 인자 — 이 랩에서 가장 조심할 부분

`use_llmlingua2=True` 인 압축기에 `question` 이나 `rank_method` 를 넘기면
**에러 없이 무시됩니다.** LongLLMLingua 설정을 v2 에 잘못 붙여도 그냥
돌아가고 결과만 v2 그대로입니다.

숫자가 안 바뀌는 이유를 찾느라 시간을 버리기 쉬워서, 어댑터가 거부합니다.

In [ ]:
try:
    L.check_params("v2", {"rate": 0.5, "question": "환불 수수료는?",
                          "rank_method": "longllmlingua"})
    print("통과 — 이러면 안 됩니다")
except ValueError as e:
    print("거부됨")
    print()
    print(e)

## 5. 이 랩의 모든 조건 돌려보기

**조건 1개 = 파일 1개**입니다. `configs/` 를 훑으면 이 랩이 답할 수 있는
질문이 전부 나옵니다. 설정을 새로 추가해도 이 셀은 고칠 필요가 없습니다.

각 조건은 `runs/04-llmlingua/<설정이름>/<시각>/` 에 따로 기록됩니다. 나중에
"그때 무엇을 돌렸나" 를 설정 이름만 보고 알 수 있게 하려는 것입니다.

| 설정 | 무엇을 보려고 |
|---|---|
| `v2` | 이 랩의 권장 조건 |
| `v1` | 질문 없이 토큰 정보량만 |
| `long` | 질문을 주면 나아지나 |
| `v2-noop` | `rate=1.0` **자가 점검** |

`v2-noop` 이 중요합니다. 아무것도 안 버리는 설정인데 **토큰이 늘어납니다.**

In [ ]:
def run_config(path):
    cfg = C.load(path)
    cs = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
    cnt = T.make_counter({"mode": "local"}, cfg.model)
    params = dict(cfg.params)
    variant = params.pop("variant", "v2")
    params.pop("model_name", None)

    run = Run(cfg, RUNS)
    for x in cs:
        after, extra = compress(x.text, question=x.question or "",
                                variant=variant, **params)
        extra["lang"] = x.meta.get("lang", "-")
        run.add(metrics.per_case(x.id, x.kind, x.text, after,
                                 x.must_include, cnt, extra),
                before=x.text, after=after)

    m = metrics.aggregate(run.records, cnt)
    m["variant"] = variant
    m["config"] = cfg.name
    m["rate"] = params.get("rate")
    for lg in ("ko", "en"):
        xs = [r for r in run.records if r["lang"] == lg]
        if xs:
            m[f"surv_{lg}"] = sum(r["survival"] for r in xs) / len(xs)
    return cfg, m, run.finish(m, [f"변형 {variant}"])


results = []
for p in sorted(Path("configs").glob("*.yaml")):
    cfg, m, out = run_config(p)
    results.append((cfg.name, m, out))
    print(f'{cfg.name:10s} rate={m["rate"]} · 절감 {m["saved"]:6.1%} · '
          f'보존 {m["survival_mean"]:6.1%}', flush=True)

## 6. 조건 비교

같은 코드에 조건만 바꿔 돌린 결과입니다. **숫자 하나가 아니라 표를 보세요.**
어떤 조건에서 무엇을 얻고 무엇을 잃는지가 이 랩의 결론입니다.

In [ ]:
n_ko = len([c for c in cases if c.meta["lang"] == "ko"])
n_en = len([c for c in cases if c.meta["lang"] == "en"])

# 뒤의 세 열은 같은 보존율을 언어별로 나눈 것입니다. 열 이름에 건수를 적어
# "전체가 두 언어의 단순평균인가?" 라는 오해를 줄입니다.
rows = []
for n, m, _ in results:
    ko, en = m.get("surv_ko"), m.get("surv_en")
    gap = (en - ko) if (ko is not None and en is not None) else None
    rows.append([n, m["variant"], m["rate"], pct(m["saved"]),
                 pct(m["survival_mean"]), pct(ko), pct(en),
                 "—" if gap is None else f"{gap * 100:+.1f}%p"])

table(
    ["설정", "변형", "rate", "절감",
     f"보존율 전체({len(cases)}건)", f"한국어({n_ko}건)", f"영어({n_en}건)",
     "영−한 격차"],
    rows,
    align=["left", "left", "right", "right", "right", "right", "right", "right"],
    title="조건 비교 — 뒤의 세 열은 같은 보존율을 언어별로 쪼갠 것입니다",
    note=f"'전체' 는 {len(cases)}건 전부의 평균입니다. 지금은 한·영이 {n_ko}건씩 "
         f"같아서 두 언어 평균의 가운데값과 일치하지만, 건수가 달라지면 "
         f"많은 쪽으로 기웁니다.",
)

real = [m for _, m, _ in results if m.get("rate") != 1.0]
if real:
    w = max(real, key=lambda m: (m.get("surv_en") or 0) - (m.get("surv_ko") or 0))
    print(f"격차가 가장 큰 조건은 {w['config']} 입니다.")
    print(f"  전체 {w['survival_mean']:.1%} 로는 무난해 보이지만 "
          f"한국어만 보면 {w['surv_ko']:.1%} 입니다.")
    print("한국어 서비스에 쓰신다면 '전체' 가 아니라 '한국어' 열을 보세요.")

by = {n: m for n, m, _ in results}
if "v2-noop" in by:
    m = by["v2-noop"]
    print(f'v2-noop: rate=1.0 인데 절감 {m["saved"]:.1%} · '
          f'보존율 {m["survival_mean"]:.1%}')
    print("아무것도 안 버렸는데 토큰이 늘었습니다. 토큰에서 텍스트를 다시")
    print("만들면서 '32,450,000' 이 '32, 450, 000' 처럼 벌어지기 때문입니다.")
if "v2" in by and "long" in by:
    print()
    print(f'v2 보존 {by["v2"]["survival_mean"]:.1%} vs '
          f'long 보존 {by["long"]["survival_mean"]:.1%} — '
          f'질문을 주는 long 이 오히려 나쁩니다.')
    print("작은 모델로 토큰 단위 프루닝을 하면 숫자가 조각나기 때문입니다.")

## 7. 압축률 스윕 — 어디서 무너지나

`rate` 는 **남길 비율**입니다. 낮출수록 많이 버립니다.
언어별로 무너지는 지점이 다른지 봅니다.

In [ ]:
rows = []
for rate in [0.9, 0.7, 0.5, 0.3]:
    recs = []
    for x in cases:
        out, meta = compress(x.text, variant="v2", rate=rate,
                             force_reserve_digit=True)
        recs.append({"lang": x.meta["lang"],
                     "s": survival(out, x.must_include),
                     "tb": counter(x.text), "ta": counter(out)})
    tb = sum(r["tb"] for r in recs)
    ta = sum(r["ta"] for r in recs)
    g = {lg: [r["s"] for r in recs if r["lang"] == lg] for lg in ("ko", "en")}
    rows.append([rate, pct(1 - ta / tb),
                 pct(sum(r["s"] for r in recs) / len(recs)),
                 pct(min(r["s"] for r in recs)),
                 pct(sum(g["ko"]) / len(g["ko"])),
                 pct(sum(g["en"]) / len(g["en"]))])
    print(f"  rate={rate} 완료", flush=True)

table(
    ["rate", "절감", f"보존 전체({len(cases)}건)", "보존 최저",
     f"한국어({n_ko}건)", f"영어({n_en}건)"],
    rows,
    align=["right"] * 6,
    title="압축률 스윕 (v2) — 언어별로 무너지는 지점이 다릅니다",
    note="rate 0.7 을 보세요. 영어는 아직 멀쩡한데 한국어는 이미 무너집니다. "
         "'전체' 는 12건 평균이라 이 격차를 가립니다.",
)

## 정리

- **세 변형 중 `v2` 만 쓸 만했습니다** — 작은 모델로 토큰 단위 프루닝을 하면
  숫자와 식별자가 조각납니다
- **한국어가 영어보다 먼저 무너집니다** — 같은 `rate` 에서 영어가 91.7% 일 때
  한국어는 66.7% 였습니다
- **한국어는 애초에 토큰을 더 씁니다** — 문자당 1.8배. 압축이 더 절실한데
  품질은 더 나쁩니다
- **`rate=1.0` 이 무손실이 아닙니다** — 토큰에서 텍스트를 재구성하면서
  숫자 서식이 벌어져 오히려 늘어납니다
- **숫자·식별자가 답인 문서에는 쓰지 마세요**

### 결론을 일반화하지 마세요

이 랩은 **0.5B 모델**로 돌립니다. 논문은 7B 를 썼습니다.
"LongLLMLingua 가 나쁘다" 가 아니라 **"작은 순위 모델로는 못 쓴다"** 입니다.
`configs/*.yaml` 의 `model_name` 으로 바꾸실 수 있습니다.